# Diviser pour régner — Exercice 8 : la paire de points la plus proche

> **Énoncé (résumé)**
> Étant donnés $n$ points du plan, sous la forme d'un tableau de tuples `(x, y)`, trouver la **plus petite distance** entre deux de ces points.

On compte les appels à `distance` avec une variable globale, pour comparer les méthodes.

In [1]:
from random import random, seed
from math import sqrt, inf
import time

seed(42)
points = [(random()*10**9, random()*10**9) for _ in range(1000)]
print(points[:3])

[(639426798.4578837, 25010755.222666938), (275029318.3691193, 223210738.14882275), (736471214.1640124, 676699487.4229113)]


## Question 1 : `distance(p, q)`

Distance euclidienne (théorème de Pythagore) : $d = \sqrt{(x_q - x_p)^2 + (y_q - y_p)^2}$.

Les coordonnées sont dans le tuple : `p[0]` est l'abscisse et `p[1]` l'ordonnée.

In [2]:
nb_appels = 0

def distance(p, q):
    global nb_appels
    nb_appels += 1
    return sqrt((q[0] - p[0])**2 + (q[1] - p[1])**2)


assert distance((0, 0), (3, 4)) == 5

## Question 2 : la force brute

### (a) `shortest_distance_bf(pts)`

On teste **tous les couples** de points et on garde le minimum.

💡 La seconde boucle commence à `i + 1` :
- on ne compare pas un point avec lui-même (la distance serait 0) ;
- on ne compare pas deux fois le même couple (`d(A, B) = d(B, A)`).

On part de `inf` (l'infini) : n'importe quelle distance sera plus petite.

In [3]:
def shortest_distance_bf(pts):
    d_min = inf
    n = len(pts)
    for i in range(n):
        for j in range(i + 1, n):
            d = distance(pts[i], pts[j])
            if d < d_min:
                d_min = d
    return d_min


nb_appels = 0
debut = time.time()
d_bf = shortest_distance_bf(points)
print(f"Plus courte distance : {d_bf:.2f}  |  appels à distance : {nb_appels}  |  {time.time() - debut:.2f} s")

Plus courte distance : 1251419.02  |  appels à distance : 499500  |  0.12 s


### (b) Nombre d'appels à `distance`

Le point d'indice 0 est comparé à $n-1$ points, le point d'indice 1 à $n-2$ points, et ainsi de suite :

$$(n-1) + (n-2) + \cdots + 1 + 0 = \frac{n(n-1)}{2}$$

Pour $n = 1000$ : $\dfrac{1000 \times 999}{2} = 499\,500$, c'est bien ce qu'on a mesuré. Le coût est **quadratique**, de l'ordre de $n^2$.

---
## Question 3 : « diviser pour régner »

### Rappel du principe
1. `px` et `py` : les points triés par **abscisse** et par **ordonnée** ;
2. on coupe `px` en deux moitiés `px1` (gauche) et `px2` (droite) ;
3. récursivement : $d_1$ = plus petite distance à gauche, $d_2$ = à droite, et $\delta = \min(d_1, d_2)$ ;
4. il reste les couples **à cheval** (un point à gauche, un point à droite). Seuls les points dont l'abscisse est entre $m - \delta$ et $m + \delta$ peuvent faire mieux que $\delta$ : c'est la **bande** ;
5. dans la bande triée par ordonnée, il suffit de comparer chaque point avec les **7 suivants** : au-delà, l'écart des ordonnées dépasse déjà $\delta$.

### (a) `get_sorted(pts)`

`sorted` avec le paramètre `key` permet de choisir **selon quoi** on trie : `key=lambda p: p[0]` trie selon l'abscisse.

In [4]:
def get_sorted(pts):
    px = sorted(pts, key=lambda p: p[0])
    py = sorted(pts, key=lambda p: p[1])
    return px, py


px, py = get_sorted([(3, 1), (1, 5), (2, 2)])
print(px, py)

[(1, 5), (2, 2), (3, 1)] [(3, 1), (2, 2), (1, 5)]


### (b) `shortest_distance_dac(px, py)` : une première version directe

On traduit le principe tel quel :
- **cas de base** : avec 3 points ou moins, on utilise la force brute (au plus 3 appels) ;
- pour obtenir `py1` et `py2` (les points de chaque moitié triés par ordonnée), on **retrie** `px1` et `px2` ;
- dans la bande, on compare **tous les couples**.

In [5]:
def shortest_distance_dac_v1(px, py):
    n = len(px)
    if n <= 3:
        return shortest_distance_bf(px)
    milieu = n // 2
    px1, px2 = px[:milieu], px[milieu:]
    m = (px1[-1][0] + px2[0][0]) / 2
    py1 = sorted(px1, key=lambda p: p[1])          # on retrie à chaque appel...
    py2 = sorted(px2, key=lambda p: p[1])
    d1 = shortest_distance_dac_v1(px1, py1)
    d2 = shortest_distance_dac_v1(px2, py2)
    delta = min(d1, d2)
    bande = [p for p in py if m - delta <= p[0] <= m + delta]
    return min(delta, shortest_distance_bf(bande))  # ... et on compare TOUS les couples de la bande


nb_appels = 0
debut = time.time()
d = shortest_distance_dac_v1(*get_sorted(points))
print(f"{d:.2f}  |  appels : {nb_appels}  |  {time.time() - debut:.3f} s  |  identique à bf : {d == d_bf}")

1251419.02  |  appels : 11966  |  0.006 s  |  identique à bf : True


### (c) Est-elle plus performante ? Optimisation

Sur des points **bien répartis**, elle est déjà beaucoup plus rapide : la bande est étroite et contient peu de points.

**Mais cette version a deux défauts :**
1. **Le pire cas :** si tous les points ont presque la même abscisse (par exemple tous sur une ligne verticale), **tous** les points tombent dans la bande. Comparer tous les couples de la bande coûte alors $n^2$, comme la force brute !
2. **Les tris répétés :** retrier `px1` et `px2` à chaque appel coûte $n\log(n)$ par niveau, alors qu'on a déjà `py`.

On le constate sur des points presque alignés verticalement :

In [6]:
seed(1)
verticaux = [(random(), random()*10**9) for _ in range(1500)]   # abscisses entre 0 et 1 seulement

for nom, f in [("force brute", lambda: shortest_distance_bf(verticaux)),
               ("dac v1", lambda: shortest_distance_dac_v1(*get_sorted(verticaux)))]:
    nb_appels = 0
    debut = time.time()
    f()
    print(f"{nom:>12} : {nb_appels:>8} appels  |  {time.time() - debut:.2f} s")

 force brute :  1124250 appels  |  0.61 s


      dac v1 :  2240360 appels  |  0.94 s


❌ La version 1 n'est **pas meilleure** que la force brute dans ce cas.

**Les deux optimisations :**
1. **La remarque cruciale de l'énoncé :** dans la bande triée par ordonnée, l'élément d'indice $i+7$ a une ordonnée supérieure de plus de $\delta$ à celle de l'élément $i$. Il est donc **inutile** d'aller plus loin : on compare chaque point avec **au plus 7 voisins**. La bande coûte alors au plus $7n$ appels au lieu de $n^2$. (On va jusqu'à $i+7$ inclus, par sécurité en cas d'égalité.)
2. **Ne plus retrier :** pour construire `py1` et `py2`, on **parcourt `py` une seule fois** (qui est déjà trié !) et on range chaque point à gauche ou à droite. Pour savoir rapidement si un point est dans `px1`, on utilise un **ensemble** (`set`) : le test `p in ensemble` est quasi instantané. Coût : $n$ par appel au lieu de $n\log(n)$.

Le coût total devient $n\log(n)$ : $\log(n)$ niveaux, avec un travail de l'ordre de $n$ à chaque niveau, exactement comme le tri fusion.

⚠️ La séparation avec un `set` suppose que **tous les points sont distincts**, ce qui est le cas avec des coordonnées tirées au hasard. Avec des doublons, la distance minimale vaut 0 et on peut la détecter à part.

In [7]:
def shortest_distance_dac(px, py):
    n = len(px)
    if n <= 3:
        return shortest_distance_bf(px)
    milieu = n // 2
    px1, px2 = px[:milieu], px[milieu:]
    m = (px1[-1][0] + px2[0][0]) / 2

    # py1 et py2 en un seul parcours de py (qui est déjà trié par ordonnée)
    gauche = set(px1)
    py1, py2 = [], []
    for p in py:
        if p in gauche:
            py1.append(p)
        else:
            py2.append(p)

    delta = min(shortest_distance_dac(px1, py1), shortest_distance_dac(px2, py2))

    # la bande, déjà triée par ordonnée puisqu'on la prend dans py
    bande = [p for p in py if m - delta <= p[0] <= m + delta]
    for i in range(len(bande)):
        for k in range(i + 1, min(i + 8, len(bande))):     # au plus 7 voisins
            d = distance(bande[i], bande[k])
            if d < delta:
                delta = d
    return delta


for nom, pts in [("points aléatoires", points), ("points verticaux", verticaux)]:
    nb_appels = 0
    debut = time.time()
    d = shortest_distance_dac(*get_sorted(pts))
    print(f"{nom:>18} : {nb_appels:>6} appels  |  {time.time() - debut:.3f} s  |  "
          f"correct : {d == shortest_distance_bf(pts)}")

 points aléatoires :  10597 appels  |  0.013 s  |  correct : True


  points verticaux :  81984 appels  |  0.044 s  |  correct : True


### Vérification et comparaison des temps

In [8]:
# Vérification sur 300 petits nuages de points
for essai in range(300):
    pts = [(random()*100, random()*100) for _ in range(2 + essai % 40)]
    assert shortest_distance_dac(*get_sorted(pts)) == shortest_distance_bf(pts)
print("300 tests réussis ✔\n")

print(f"{'n':>7} | {'force brute':>11} | {'dac optimisée':>13}")
for n in [500, 1000, 2000, 20000, 100000]:
    pts = [(random()*10**9, random()*10**9) for _ in range(n)]
    if n <= 2000:
        debut = time.time()
        shortest_distance_bf(pts)
        t_bf = f"{time.time() - debut:.3f} s"
    else:
        t_bf = "trop long"
    debut = time.time()
    shortest_distance_dac(*get_sorted(pts))
    print(f"{n:>7} | {t_bf:>11} | {time.time() - debut:>11.3f} s")

300 tests réussis ✔

      n | force brute | dac optimisée


    500 |     0.086 s |       0.006 s


   1000 |     0.248 s |       0.010 s


   2000 |     0.911 s |       0.023 s


  20000 |   trop long |       0.383 s


 100000 |   trop long |       2.002 s


💡 **À retenir**
- Force brute : $\dfrac{n(n-1)}{2}$ appels, donc un coût en $n^2$.
- « Diviser pour régner » bien implémenté : coût en $n\log(n)$. Avec 100 000 points, la force brute demanderait environ 5 milliards d'appels à `distance` !
- Une idée « diviser pour régner » ne suffit pas : **l'étape de combinaison** (ici la bande) doit coûter au plus $n$ à chaque niveau. Sinon, on perd tout le gain.